# Data reconnaissance (00) - re-run

**This notebook has been rebuilt.** Its first version was recorded as done
with the calibration unverified against the real `laps.csv`. 03b was the
first stage to read the frozen dials rather than assume them, refused them,
and used a stand-in instead. What follows is the re-run, and it is the stage
that decides whether this project demonstrates Daytona and Le Mans or an
invented six-hour race.

**Boundary constraint.** Constants only; no simulation code. No change to the
`ClassDials` schema or to `ASSUMED_FIELDS` - a new or renamed field
invalidates every saved config, bank and checkpoint in the project. No engine
changes, no strategy, benchmark or agent work.

**Verification gate.** The stage was specified without one, which is how the
fault survived. Four conditions, all of which could fail and one of which is
required to: Part 6.

## What was wrong

`imsa.json` described a 216-hour race with 149 cars carrying DPi, GTLM,
GTDPRO and GTP in one field. Two faults, and the second was not suspected:

1. **The queries were not scoped to one running.** `build_race_config` took an
   ILIKE pattern on `event`, and `event` carries a circuit and no edition. Six
   Daytonas and three Le Mans went into one config: durations added, counts
   summed, stints concatenated.
2. **`car` was read as an integer, so a leading zero was lost.** `#7` Toyota
   and `#007` Aston are both Hypercar at Le Mans; `#4` Corvette and `#04`
   CrowdStrike are both at Daytona. Collapsed onto one identifier, two cars'
   laps merged - which is why every edition reported 48 hours of running for a
   24-hour race, and why Le Mans 2026's Hypercar "winner" showed 62 stops.

The two compose exactly. Summed across the Le Mans editions as the old code
pooled them: 24.1 + 48.1 + 48.2 = 120.4 hours, against the 120.4 in
`wec.json`. Daytona reconciles the same way at 216.2. The near-integer
multiple of 24 hours that looked like a clean diagnosis was an integer number
of *car-races*, not of editions.

Three further defects were found that scoping does not fix, and all three are
corrected here: `stint_number` counts driver stints rather than fuel stints;
the pit column carries hour-long repairs beside 80-second stops within a
single race; and the caution flag test swept in the chequered lap.

### Setup

In [40]:
import sys
from pathlib import Path


def find_project_root(marker: str = "src/endurance") -> Path:
    """Walk up from the working folder until the project appears."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker!r} at or above {here}.")


ROOT = find_project_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd                                          # noqa: E402
from endurance import calibrate                              # noqa: E402
from endurance import gate00                                 # noqa: E402

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

LAPS_CSV    = ROOT / "data" / "raw" / "laps.csv"
DRIVERS_CSV = ROOT / "data" / "raw" / "drivers.csv"
PARAMS_DIR  = ROOT / "data" / "processed"
PARAMS_DIR.mkdir(parents=True, exist_ok=True)

# The two races the demonstration is about, and the class each is headlined
# by. The edition is resolved from the data in Part 1 rather than named here.
ANCHORS = {
    "imsa": {"event": "%daytona%", "headline_class": "GTP",      "name": "Daytona 24"},
    "wec":  {"event": "%le mans%", "headline_class": "HYPERCAR", "name": "Le Mans 24"},
}

con = calibrate.connect(str(LAPS_CSV), str(DRIVERS_CSV))
print(f"loaded {con.execute('SELECT COUNT(*) FROM laps').fetchone()[0]:,} laps")

loaded 1,658,803 laps


## Part 1 - which race, and which running of it

`event` names a circuit. The edition discriminator is `session_id`: 1013
values, none of which spans an event, a year, a series or a session type.
`(series, event, year)` is *not* sufficient - the Asian Le Mans double-headers
put two races under one key - which is why the scope is the session and not
the year.

The table below is the one the edition decision was taken on. Note what it
says about caution share: it moves by nearly a factor of two between adjacent
Daytonas, so a dial calibrated from one running is a sample of one. Freezing
the latest edition is the decision; showing the spread beside it is what stops
the number being read as a property of the race rather than of one race.

In [41]:
RACES = {}
for series, anchor in ANCHORS.items():
    editions = calibrate.list_races(con, series, anchor["event"])
    RACES[series] = calibrate.find_race(con, series, anchor["event"])
    print(f"--- {series} {anchor['name']}: {len(editions)} editions in file, "
          f"taking {RACES[series]['label']}")
    print(editions.to_string(index=False))

--- imsa Daytona 24: 6 editions in file, taking Daytona 2026 (imsa session 682)
 session_id  year   event  cars  laps_recorded  duration_s
        388  2021 Daytona    49          33029   86301.675
        467  2022 Daytona    61          37597   86301.199
        524  2023 Daytona    61          39736   86299.507
        579  2024 Daytona    59          36628   86300.877
        634  2025 Daytona    61          37885   86300.521
        682  2026 Daytona    60          36756   86296.368
--- wec Le Mans 24: 3 editions in file, taking Le Mans 2026 (wec session 1000)
 session_id  year   event  cars  laps_recorded  duration_s
        766  2022 Le Mans    32          11514   86160.513
        944  2025 Le Mans    62          20182   86175.991
       1000  2026 Le Mans    62          20163   86176.366


In [42]:
# How far a dial moves between runnings of the same race. The caution share
# is the one that moves most, and it is the one every strategy result is
# sensitive to.
spread = []
for series, anchor in ANCHORS.items():
    for _, ed in calibrate.list_races(con, series, anchor["event"]).iterrows():
        sid = int(ed["session_id"])
        try:
            c = calibrate.calibrate_cautions(con, sid)
        except Exception as exc:                       # a partial edition
            print(f"{series} {int(ed['year'])}: skipped ({exc})")
            continue
        spread.append({"series": series, "year": int(ed["year"]),
                       "cars": int(ed["cars"]),
                       "duration_h": round(ed["duration_s"] / 3600, 2),
                       "caution_rate": round(c["caution_rate"], 3),
                       "caution_dur_s": round(c["caution_mean_dur_s"]),
                       "episodes": c["n_caution_episodes"],
                       "frozen": sid == RACES[series]["session_id"]})
spread = pd.DataFrame(spread)
spread

,series,year,cars,duration_h,caution_rate,caution_dur_s,episodes,frozen
0,imsa,2021,49,23.97,0.127,912,12,False
1,imsa,2022,61,23.97,0.228,1155,17,False
2,imsa,2023,61,23.97,0.160,988,14,False
3,imsa,2024,59,23.97,0.154,884,15,False
4,imsa,2025,61,23.97,0.179,1101,14,False
5,imsa,2026,60,23.97,0.355,3404,9,True
6,wec,2022,32,23.93,0.009,793,1,False
7,wec,2025,62,23.94,0.038,820,4,False
8,wec,2026,62,23.94,0.080,1718,4,True


## Part 2 - the five dials

One query per dial per class, every number traceable to a named function in
`src/endurance/calibrate.py`. Three of the five changed in this re-run:

- **Stint length** comes from the pit records, not from `stint_number`. Every
  step of that counter coincides with a driver change and none with a stop -
  two to three fuel stints to the step - so reading it as a fuel stint
  reported 58-lap green stints at Daytona where the winner averaged 23.
- **Pit cost** is a median, under an unchanged field name. Within one scoped
  race the column still runs to 6,800 seconds against a median of 80, so the
  arithmetic mean sat two to four times its own median and the standard
  deviation sat above the mean. The spread is the standard deviation of the
  sample trimmed at three times the median.
- **Cautions** are calibrated once for the race rather than once per class.
  They are a property of the race; calling the function inside the class loop
  is what left seven classes of one race carrying seven different episode
  lengths.

In [43]:
configs = {}
for series, anchor in ANCHORS.items():
    race = RACES[series]
    configs[series] = calibrate.build_race_config(
        con, series, race["session_id"], f"{anchor['name']} {race['year']}")
    cfg = configs[series]
    print(f"{series}: {len(cfg.classes)} classes, {cfg.total_cars} cars, "
          f"{cfg.duration_s / 3600:.2f} h - {cfg.classes[0].source_event}")

pd.concat([calibrate.dials_table(cfg) for cfg in configs.values()],
          ignore_index=True)

imsa: 4 classes, 60 cars, 23.97 h - Daytona 2026 (imsa session 682)
wec: 3 classes, 62 cars, 23.94 h - Le Mans 2026 (wec session 1000)


,series,class,base_pace_s,deg_s_per_lap,caution_rate,caution_dur_s,stint_laps,pit_s,pit_sd_s,cars
0,imsa,GTD,108.40,0.0091,0.353,3388.0,30.0,90.5,19.6,21
1,imsa,GTDPRO,108.15,0.0069,0.353,3388.0,31.0,90.5,22.4,15
2,imsa,LMP2,102.04,0.0038,0.353,3388.0,24.0,90.6,25.3,13
3,imsa,GTP,98.08,0.0014,0.353,3388.0,30.0,89.8,16.9,11
4,wec,LMGT3,237.69,0.0315,0.088,1271.0,10.0,78.7,15.2,25
5,wec,LMP2,219.77,-0.0327,0.088,1271.0,11.0,88.6,16.9,19
6,wec,HYPERCAR,208.80,-0.0322,0.088,1271.0,12.0,77.0,12.7,18


### What is measured, and what is assumed

Unchanged in this re-run, and printed here so nothing gets quietly promoted
from guess to fact. `pit_transit_frac` is on the assumed list and Part 5
measures a candidate for it without moving it.

In [44]:
from endurance import ClassDials                             # noqa: E402

sample = configs["imsa"].classes[0]
pd.DataFrame(
    [{"field": f, "value": round(getattr(sample, f), 4), "status": "measured"}
     for f in sample.measured_fields()]
    + [{"field": f, "value": getattr(sample, f), "status": "ASSUMED"}
       for f in ClassDials.assumed_fields()])

,field,value,status
0,base_pace_s,108.3980,measured
1,deg_slope_s_per_lap,0.0091,measured
2,pace_spread_s,0.5383,measured
3,lap_noise_s,0.6625,measured
4,caution_rate,0.3528,measured
5,caution_mean_dur_s,3388.2536,measured
6,green_stint_laps,30.0000,measured
7,fuel_per_lap,0.0333,measured
8,fuel_per_lap_caution,0.0200,measured
9,pit_time_mean_s,90.5030,measured


## Part 3 - degradation, and what this data can support

Degradation is fitted jointly against tyre age and laps since the last fill.
Within a stint the car gets lighter as the tyres get older and the two effects
have opposite signs, so a slope on tyre age alone is their sum. The
field-relative frame does not rescue it: cars in a class stagger their stops,
so at a given lap number they are at different points of the tank and fuel is
not common-mode.

Where a class changes tyres at every stop the two regressors are the same
number and nothing can separate them. That is reported rather than resolved:
`identified` is the column to read before the slope.

In [45]:
for series, cfg in configs.items():
    print(f"--- {series}")
    print(calibrate.degradation_table(
        con, RACES[series]["session_id"],
        [c.class_name for c in cfg.classes]).to_string(index=False))

--- imsa
 class  deg_s_per_lap  simple_slope fuel_s_per_lap  age_fuel_corr  identified  n_laps
   GTD        0.00912       0.00912           None          0.919       False    4086
GTDPRO        0.00694       0.00694           None          0.958       False    3208
  LMP2        0.00379       0.00379           None          0.970       False    2728
   GTP        0.00135       0.00135           None          0.963       False    2695
--- wec
   class  deg_s_per_lap  simple_slope  fuel_s_per_lap  age_fuel_corr  identified  n_laps
   LMGT3        0.03154       0.01996        -0.01262          0.873        True    2936
    LMP2       -0.03270      -0.03270             NaN          0.947       False    2743
HYPERCAR       -0.03217      -0.03217             NaN          0.923       False    2644


**Read the `identified` column first.** A negative slope
in a class where it is false is not a defect to be tuned away; it is the net
within-stint pace trend, which is what the engine will reproduce, and it says
that this file cannot separate tyre wear from fuel burn for that class. A
negative slope where `identified` is true would be a second defect and gate
condition four would be right to fail on it.

## Part 4 - the caution units, old and new

Carried forward from 02a. The share and the episode length are measured in
seconds of race time rather than laps, and the observed caution pace
multiplier is reported beside the assumed one - a measured counterpart to an
assumed dial, shown rather than silently substituted.

In [46]:
for series, cfg in configs.items():
    print(f"--- {series}")
    print(calibrate.caution_report(
        con, RACES[series]["session_id"],
        cfg.classes[0].base_pace_s).to_string(index=False))

--- imsa
                             quantity  legacy (laps)  measured (seconds)
                        caution share       0.227273            0.352801
                      mean episode, s    1927.075556         3388.253556
                             episodes       9.000000            9.000000
caution pace multiplier (assumed 1.6)            NaN            1.870591
               red-flag laps excluded       0.000000            0.000000
--- wec
                             quantity  legacy (laps)  measured (seconds)
                        caution share       0.044737            0.079553
                      mean episode, s    1010.184625         1718.197500
                             episodes       4.000000            4.000000
caution pace multiplier (assumed 1.6)            NaN            1.787164
               red-flag laps excluded       0.000000            0.000000


## Part 5 - `pit_transit_frac`, measured

03b established that 02c's `splash_and_dash` result depends on this dial:
gained falls monotonically as it runs 0.25 to 0.65 while no other roster row
moves monotonically. It is in `ASSUMED_FIELDS` at 0.25 and had never been
swept when 02c relied on it.

A low quantile of `pit_time` for one properly scoped race is a stop with
almost no service in it, which is the lane transit delta itself. Two things
had to be true for that to be worth measuring, and both now are: the column
had to be lane-to-lane time lost rather than stationary time - the recon put
the ratio of `pit_time` to the lap's excess over a green lap at 1.01 to 1.16
at the lower quartile - and the outliers had to be gone.

The regulations supply a cross-check and not the answer. The pit lane speed
limit is 60 km/h in both series (IMSA art. 32.3, WEC art. 12.1.4) and neither
rulebook carries the lane length, which is a circuit fact.

**This does not move the dial.** Promoting a field out of `ASSUMED_FIELDS` is
a blueprint amendment and a separate decision; what follows is the evidence
for taking it.

In [47]:
rows = []
for series, cfg in configs.items():
    for c in cfg.classes:
        pit = calibrate.calibrate_pit(con, RACES[series]["session_id"], c.class_name)
        rows.append({
            "series": series, "class": c.class_name,
            "median_stop_s": round(pit["pit_time_mean_s"], 1),
            "raw_mean_s": round(pit["pit_time_raw_mean_s"], 1),
            "trimmed_out": pit["n_pit_stops_trimmed_out"],
            "n": pit["n_pit_stops"],
            "transit_s_p05": round(pit["pit_lane_transit_s"], 1),
            "implied_frac": round(pit["pit_lane_transit_s"] / pit["pit_time_mean_s"], 3),
            "assumed_frac": c.pit_transit_frac})
pd.DataFrame(rows)

,series,class,median_stop_s,raw_mean_s,trimmed_out,n,transit_s_p05,implied_frac,assumed_frac
0,imsa,GTD,90.5,107.3,4,244,46.0,0.509,0.25
1,imsa,GTDPRO,90.5,110.8,3,186,45.7,0.505,0.25
2,imsa,LMP2,90.6,126.6,7,255,45.8,0.505,0.25
3,imsa,GTP,89.8,111.0,2,178,59.4,0.662,0.25
4,wec,LMGT3,78.7,97.3,12,661,68.5,0.871,0.25
5,wec,LMP2,88.6,104.5,5,569,81.8,0.923,0.25
6,wec,HYPERCAR,77.0,104.8,8,510,66.8,0.867,0.25


`raw_mean_s` beside `median_stop_s` is the argument for
decision 6 in one column: the mean is what the old dial reported and it is not
a stop anyone made.

`implied_frac` against `assumed_frac` is the finding. Where it lands well
above 0.25, 02c's `splash_and_dash` result was computed at the wrong end of
the dial it is most sensitive to, and 03b's sweep already tells us which
direction that moves it.

## Part 6 - the verification gate

Four conditions. Condition three is the falsifier: it widens the scope to two
adjacent editions on purpose and **requires** conditions one and two to fail.
Adjacent rather than distant, because two runnings of the same race are
genuinely similar and are the hardest case for a gate to catch.

Two things the gate does not do, stated rather than discovered:

- **Grid size does not detect pooling.** Car numbers recur between editions -
  six Daytonas carry 91 numbers, not 360 - so the count grows by far less than
  the racing does. That check earns its place against the leading-zero
  collision and against a class list assembled from somewhere other than the
  scope. Duration and lap counts are what catch pooling.
- **Nor does the pit dial.** Since it became a median it is robust to pooling
  in the same way base pace always was: the edition with more stops carries
  the statistic. Both are better dials for it and neither is a detector.

In [48]:
gates = {}
for series in ANCHORS:
    editions = calibrate.list_races(con, series, ANCHORS[series]["event"])
    others = [int(s) for s in editions["session_id"]
              if int(s) != RACES[series]["session_id"]]
    print(f"===== {series}: {RACES[series]['label']}, "
          f"falsifier pools with session {others[-1]}")
    gates[series] = gate00.run_gate(con, series, RACES[series]["session_id"],
                                    others[-1], cfg=configs[series])
    print()

===== imsa: Daytona 2026 (imsa session 682), falsifier pools with session 634
                       condition                                                       check                  value                                   threshold  passed
       one: the race is one race                                       duration vs scheduled                  23.97                                    24 h ±2%    True
       one: the race is one race                 one number, one car (grid is a single grid)                   60.0                                          60    True
       one: the race is one race                                        classes all competed                    0.0                                           0    True
       one: the race is one race                                   GTD laps within the clock                  661.0                                         836    True
       one: the race is one race                                GTDPRO laps within

In [49]:
# The falsifier's own detail: every one of these is required to fail.
for series, g in gates.items():
    detail = g[g["condition"].str.startswith("three: falsifier (")]
    print(f"--- {series}: {int((~detail['passed']).sum())} of {len(detail)} "
          f"pooled checks failed, as required")
    print(detail[["check", "value", "threshold", "passed"]].to_string(index=False))
    print()

--- imsa: 6 of 23 pooled checks failed, as required
                                      check          value threshold  passed
                      duration vs scheduled          47.94  24 h ±2%   False
one number, one car (grid is a single grid)           74.0        72   False
                       classes all competed            0.0         0    True
                  GTD laps within the clock          719.0      1664    True
               GTDPRO laps within the clock          723.0      1671    True
                 LMP2 laps within the clock          765.0      1787    True
                  GTP laps within the clock          781.0      1847    True
      GTD stint dial inside observed stints           29.0     28-36    True
           GTD pit dial inside observed IQR           91.4 89.6-93.8    True
              GTD pit sd not above its mean           18.2      91.4    True
                GTD winning laps, 0 to +10% 1449 (+101.5%)       719   False
   GTDPRO stint dial ins

In [50]:
for series, g in gates.items():
    print(f"{series}: gate {'PASSES' if gate00.gate_passes(g) else 'FAILS'}")

imsa: gate PASSES
wec: gate PASSES


### Where the stint discrepancy lives

Condition two's stint rows fail in **opposite directions** in the two series -
IMSA long, WEC short - which rules out a single systematic cause. Three
quantities side by side: what the file's stints look like, what the dial took
from them, and what the engine then does with it.

`dial_green_stint_laps` above `file_green_max` would be a tank nobody ever
emptied, and the upper-quartile choice would be wrong for this purpose.
`sim_laps_per_stop` far from the dial puts the discrepancy in the engine's
stopping rule or in `fuel_per_lap_caution` - an assumed dial, never swept -
rather than in the calibration. Those have different owners, and the gate
should not be rewritten until it is known which one this is.

In [51]:
for series, cfg in configs.items():
    print(f"--- {series}")
    print(gate00.stint_diagnostic(
        con, cfg, RACES[series]["session_id"]).to_string(index=False))
    print()

--- imsa
 class  file_green_median  file_green_q75_the_dial  file_green_max  dial_green_stint_laps  file_laps_per_stop  sim_laps_per_stop  caution_rate  fuel_per_lap_caution_ratio
   GTD               28.0                     30.0              36                   30.0                26.4               31.1         0.353                         0.6
GTDPRO               28.0                     31.0              34                   31.0                27.6               31.0         0.353                         0.6
  LMP2               22.0                     24.0              26                   24.0                21.4               24.7         0.353                         0.6
   GTP               29.0                     30.0              32                   30.0                23.5               31.3         0.353                         0.6

--- wec
   class  file_green_median  file_green_q75_the_dial  file_green_max  dial_green_stint_laps  file_laps_per_stop  sim_laps_per_s

## Part 7 - freeze

`data/processed/{series}.json` is what every later stage reads. Writing it is
the last thing this notebook does, and it happens only if the gate passed -
a config that fails its own gate must not be able to reach 02b's bank or
03b's training run.

Everything downstream re-runs from here. The seed lists do not move -
`draw_seed_bank` draws from `draw_seed` alone - but the races they name do,
because a seed plus different dials is a different race.

In [52]:
for series, cfg in configs.items():
    if not gate00.gate_passes(gates[series]):
        print(f"{series}: gate failed, NOT written")
        continue
    path = PARAMS_DIR / f"{series}.json"
    cfg.save(path)
    print(f"wrote {path}  ({cfg.classes[0].source_event})")

wrote /Users/joshzola/Documents/motorsport/endurance racing/endurance strategy rl/data/processed/imsa.json  (Daytona 2026 (imsa session 682))
wrote /Users/joshzola/Documents/motorsport/endurance racing/endurance strategy rl/data/processed/wec.json  (Le Mans 2026 (wec session 1000))


## Where this leaves us

**The stage.** Both configs describe one running of one race, scoped by
`session_id`, with `car` read as text so a leading zero is a different car.

**What must re-run, in order.** `scripts/freeze_assets.py --force`; both
policies retrained, which `PolicyCard.check` will insist on against the new
`dials_fingerprint`; 02c's roster table; 03b's tables; and 01 Part 6 and 02a
Part 6, outstanding since 02a and now runnable against real dials.

**What does not.** Nothing in 03b's findings: every number it produced is
labelled stand-in, and the degenerate reward, the `pit_transit_frac` exploit
and the verification gates are about the apparatus rather than about the
dials.

**What is still open.** Where `identified` is false in Part 3, this file
cannot separate tyre wear from fuel burn and the dial holds a net trend. That
is a documented limitation, not a defect, and it is the honest outcome the
re-run was allowed to have.